In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import polars as pl
from polars import col, lit, when
import re

import sys
import os

root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from scripts.feature_calculation import build_processed_dataset
from scripts.feature_engineering import compute_global_stats
from scripts.train_eval_model import train_baseline

### Creating train dataframe with new features

При считывании создаем колонку, по которой можно отличить train от pretrain и приводим данные к одинаковой схеме

In [3]:
train_1 = pl.scan_parquet('../../data/train_part_1.parquet')
train_1 = train_1.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_1 = pl.scan_parquet('../../data/pretrain_part_1.parquet')
pretrain_1 = pretrain_1.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

train_2 = pl.scan_parquet('../../data/train_part_2.parquet')
train_2 = train_2.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_2 = pl.scan_parquet('../../data/pretrain_part_2.parquet')
pretrain_2 = pretrain_2.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

train_3 = pl.scan_parquet('../../data/train_part_3.parquet')
train_3 = train_3.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_3 = pl.scan_parquet('../../data/pretrain_part_3.parquet')
pretrain_3 = pretrain_3.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

pretest = pl.scan_parquet('../../data/pretest.parquet')
pretest = pretest.with_columns(
    pl.lit(0).alias('is_train')
)
test = pl.scan_parquet('../../data/test.parquet')
test = test.with_columns(
    pl.lit(1).alias('is_train')
)

labels = pl.scan_parquet('../../data/train_labels.parquet')

In [4]:
full_train = pl.concat([pretrain_1, pretrain_2, pretrain_3, 
                        train_1, train_2, train_3, pretest, test], how='vertical') 

In [5]:
import shutil

# Compute population-level statistics from the full pre-test history
# (pretrain + train). This covers all data available before the test period,
# giving the most stable estimates for global frequencies and amount
# distributions. The resulting tables are frozen here and injected into
# build_processed_dataset so that Sections F and G use proper training-set
# statistics instead of falling back to per-customer cumulative proxies.
#
# labels_lf is passed so that compute_global_stats can also compute
# Bayesian-smoothed target encodings for event_type_nm, event_desc,
# channel_indicator_type, channel_indicator_sub_type, the
# (event_type_nm, event_desc) pair, and the two within-group channel
# encodings.  Without it those 7 features default to the constant 0.5.
#
# Note: pretest and test are intentionally excluded — they contain transactions
# from the test period and must not influence the population-level statistics
# used to score those same transactions.
history_lf = pl.concat([pretrain_1, pretrain_2, pretrain_3,
                         train_1,    train_2,    train_3])
global_stats = compute_global_stats(history_lf, labels_lf=labels)

# Clear the existing output so that partitions built without global_stats
# do not persist alongside the newly generated ones.
shutil.rmtree('../data_processed/', ignore_errors=True)
shutil.rmtree('../data_splits/', ignore_errors=True)

build_processed_dataset(full_train, global_stats=global_stats)

  100,000 customers → 50 partitions × ~2,000 customers each
[██████████████████████████████] 100.0%  part 50/50  (1,707,096 rows)  elapsed 2m18s  ETA 0s                

Done. 86,311,523 total rows written across 50 files in '../data_processed/'.


Посмотрим что получилось

In [6]:
example_data = pl.read_parquet('../data_processed/part_0000.parquet')
example_data.head()

customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,accept_language,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,is_train,hour,day_of_week,day_of_month,week_of_year,hour_of_day,is_weekend,is_night,is_working_hour,minutes_from_midnight,log_amount,amount_abs,amount_round_100,amount_round_1000,…,amount_mean_pos_user,amount_std_pos_user,pos_last_seen_days,amount_zscore_given_pos,pos_spend_vs_90d,spend_in_tx_type_lifetime,tx_count_in_tx_type_lifetime,amount_mean_tx_type_user,amount_std_tx_type_user,tx_type_usage_share,amount_zscore_given_tx_type,tx_type_spend_share,tx_type_spend_vs_90d,is_new_channel_desc_combo,is_new_channel_type_combo,is_new_subchannel_type_combo,is_new_type_desc_combo,is_new_txtype_channel_combo,amount_zscore_event_desc_global,amount_zscore_event_type_global,amount_zscore_subchannel_global,amount_zscore_pos_global,global_subchannel_freq,event_type_nm_target_enc,event_desc_target_enc,channel_type_target_enc,channel_subtype_target_enc,mcc_target_enc,type_desc_pair_target_enc,channel_type_fraud_rate_within_group,channel_subtype_fraud_rate_within_group,is_very_high_risk_desc,is_high_risk_desc,is_low_risk_desc,is_high_risk_channel,is_p2p_danger_channel,is_near_certain_fraud_pair
i64,i64,datetime[μs],i32,i32,i32,i32,f32,i32,str,i32,str,str,i32,i64,i32,str,str,str,str,i32,i32,str,i32,i8,i8,i8,i8,i8,i8,i8,i8,i8,f32,f32,i8,i8,…,f32,f32,i32,f32,f32,f32,u32,f32,f32,f32,f32,f32,f32,i8,i8,i8,i8,i8,f32,f32,f32,f32,u32,f32,f32,f32,f32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8
123123123124290,123939168233221,2024-10-01 00:03:20,7,56,3,4,null,null,null,null,"""ru""",null,3,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,3,0.0,0.0,1,1,…,3.8061e6,3.4431976e7,0,-0.110539,32.994102,0.0,342,0.0,0.0,0.554295,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,-0.018163,0.0,30480551,0.581235,0.578522,0.510886,0.511365,0.587769,0.581235,0.55867,0.558889,0,0,0,0,0,0
123140302994545,123861859323955,2024-10-01 00:03:20,7,56,4,15,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,3,0.0,0.0,1,1,…,375927.78125,3.3439e6,0,-0.112423,4.654504,0.0,540,0.0,0.0,0.631579,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,-0.012095,0.0,80798395,0.581235,0.578522,0.549743,0.550141,0.587769,0.581235,0.573346,0.573346,0,0,0,0,0,0
123131713060513,126533330625689,2024-10-01 00:05:46,14,75,6,5,23267.0,0,"""4""",3,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,5,10.054834,23267.0,0,0,…,64535.511719,68930.75,1,-0.598695,0.23227,1.6201709e7,268,60454.136719,68542.4375,0.208075,-0.542542,0.145207,0.294504,0,0,0,0,0,-0.046248,-0.008659,-0.078469,-0.113521,52449209,0.649522,0.730438,0.813503,0.702705,0.525717,0.730438,0.801452,0.706026,0,0,0,1,0,0
123123123126335,125107399907706,2024-10-01 00:07:13,14,75,6,5,22498.0,0,null,3,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,7,10.021226,22498.0,0,0,…,114889.53125,951373.125,0,-0.097114,0.988995,1.93526752e8,452,428156.5,2.288091e6,0.267614,-0.177291,0.587681,3.552072,0,0,0,0,0,-0.046659,-0.008663,-0.078751,-0.113704,52449209,0.649522,0.730438,0.813503,0.702705,0.587769,0.730438,0.71986,0.902357,0,0,0,1,1,0
123131713060293,126181143683685,2024-10-01 00:09:25,7,56,3,4,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,9,0.0,0.0,1,1,…,234059.890625,1.0091e6,0,-0.231944,4.519429,0.0,559,0.0,0.0,0.35047,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,-0.018163,0.0,30480551,0.581235,0.578522,0.510886,0.511365,0.587769,0.581235,0.55867,0.558889,0,0,0,0,0,0


### Training baseline on engineered features

In [3]:
train_baseline()

Found 50 parquet partitions in '../data_processed'

Ensemble seeds   : [42, 7, 13]
CatBoost weight  : 0.25

  Cache hit — reusing memmap files from '../data_splits'
    X_train       : 60,916,817 × 401
    X_val         : 24,761,023 × 401
    Labeled in val: 26,865 / 24,761,023
    Features      : 401  (event_type_nm … is_near_certain_fraud_pair)

  Feature columns : 401
  X_train shape   : (60916817, 401)
  X_val shape     : (24761023, 401)


  Group 0 — NP_TYPE7  ensemble

  Train rows  : 24,993,804  (12,995 positives)
  Val rows    : 10,704,571  (9,692 labeled,  4,693 positives)


  ── LGBM seed 1/3 (seed=42) ──────────────────────────────────
  Undersampling: kept 12,995 pos + 1,249,040 neg (1:96 ratio, 1,262,035 total rows)
  Labeled val rows for early stopping: 9,692 / 9,692

Training LightGBM …
  objective                : binary
  metric                   : average_precision
  verbosity                : -1
  device_type              : cpu
  num_threads              : 7
  num_le

In [8]:
sub = pd.read_csv('submission.csv')
sub.shape

(633683, 2)

In [9]:
sub.head()

,event_id,predict
0,125390866897300,0.538617
1,126189731373139,0.429353
2,125081630705881,0.422159
3,125262020316273,0.333167
4,125682927274458,0.336621


### Feature importance analysis

(Устарело)

In [11]:
import lightgbm as lgb

bst = lgb.Booster(model_file='baseline_lgbm.txt')

In [22]:
model = lgb.Booster(model_file='baseline_lgbm.txt')

In [23]:
model.feature_name()

['event_type_nm',
 'event_desc',
 'channel_indicator_type',
 'channel_indicator_sub_type',
 'operaton_amt',
 'currency_iso_cd',
 'pos_cd',
 'timezone',
 'session_id',
 'operating_system_type',
 'phone_voip_call_state',
 'web_rdp_connection',
 'hour',
 'day_of_week',
 'day_of_month',
 'week_of_year',
 'hour_of_day',
 'is_weekend',
 'is_night',
 'is_working_hour',
 'minutes_from_midnight',
 'log_amount',
 'amount_abs',
 'amount_round_100',
 'amount_round_1000',
 'amount_is_integer',
 'amount_missing_flag',
 'amount_currency_mismatch_flag',
 'amount_usd_normalized',
 'is_month_start',
 'is_month_end',
 'is_payday',
 'is_holiday',
 'tx_count_lifetime',
 'time_since_last_tx_minutes',
 'mcc_freq_user_cum',
 'pos_freq_user_cum',
 'event_desc_user_freq',
 'event_type_user_freq',
 'mcc_is_new_for_user',
 'pos_cd_is_new',
 'mcc_transaction_share_user',
 'pos_cd_transaction_share_user',
 'merchant_switch_flag',
 'time_since_last_3_tx_mean',
 'mcc_frequency_user',
 'event_desc_is_new_for_user',
 '

In [24]:
importances = importances = model.feature_importance(importance_type='gain')
feature_importance_df = pd.DataFrame({
    'Feature': model.feature_name(),
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [25]:
feature_importance_df.head(30)

,Feature,Importance
44,time_since_last_3_tx_mean,137966.299146
213,global_mcc_freq,90486.215285
37,event_desc_user_freq,90138.605416
190,spend_in_channel_lifetime,61330.408890
220,global_event_desc_freq,59363.082040
4,operaton_amt,58881.362158
237,amount_zscore_given_device,58240.613001
225,time_gap_mean_30d,54442.208632
48,event_desc_share_user,49832.267904
239,amount_zscore_channel,45464.942554


In [26]:
feature_importance_df.to_csv('feature_importances.csv', index=False)